# 디스플레이 공정 자료의 전처리와 탐색

- LG Aimers 5기 제품 이상 여부 판별 과제의 전처리와 탐색적 자료 분석
- 공정 4단계(Dam, AutoClave, Fill1, Fill2)별 변수 분포, 완전 상관 변수, 정상과 불량 집단의 분포 차이 확인

- 필요한 파일: `train.csv`, `test.csv` (대회 제공 자료, 비공개)
- 순서: 밀린 열 복구 → 범주 인코딩 → 공정 간 값 불일치 확인 → 공정별 탐색(분포, 상관, 결과변수와의 관계)

## 1. 준비

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from pycaret.classification import *

from scipy.stats import chi2_contingency
from statsmodels.tools.tools import add_constant
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

In [ ]:
RANDOM_STATE = 110

# 대회 제공 자료
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
test_df = test_df.drop(['Set ID', 'target'], axis=1)  # 식별 번호와 빈 결과변수 열 제외

## 2. 전처리

### 2.1. 밀린 열 복구

- 일부 행은 특정 변수부터 값이 한 칸씩 밀려 기록되어 있어, 밀린 값을 제자리로 옮김
- X 좌표의 "OK"는 결측으로 처리

In [ ]:
# X 좌표의 "OK"를 결측으로 변환

columns_to_replace_ok = [
    'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
    'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
    'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'
]

# 학습 자료
for column in columns_to_replace_ok:
    train_df[column] = train_df[column].replace('OK', np.nan).astype(float)

# 시험 자료
for column in columns_to_replace_ok:
    test_df[column] = test_df[column].replace('OK', np.nan).astype(float)

In [ ]:
def shift_and_fill(data, start_col, end_col, step):
    # start_col이 결측인 행: start_col~end_col 구간 값을 step 칸씩 앞으로 당기고 마지막 칸은 결측

    start_idx = data.columns.get_loc(start_col)

    end_idx = data.columns.get_loc(end_col)

    missing_indices = data[data[start_col].isnull()].index

    for idx in missing_indices:
        current_idx = start_idx

        while current_idx <= end_idx - step:
            next_idx = current_idx + step
            data.iloc[idx, current_idx] = data.iloc[idx, next_idx]
            current_idx = next_idx

        data.iloc[idx, end_idx] = np.nan

In [ ]:
# Dam
shift_and_fill(train_df, 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
               'WorkMode Collect Result_Dam', 3)
shift_and_fill(test_df, 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
               'WorkMode Collect Result_Dam', 3)
# Fill1
shift_and_fill(train_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1",
               'WorkMode Collect Result_Fill1', 3)
shift_and_fill(test_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1",
               'WorkMode Collect Result_Fill1', 3)

# Fill2
shift_and_fill(train_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2",
               'WorkMode Collect Result_Fill2', 3)
shift_and_fill(test_df, "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2",
               'WorkMode Collect Result_Fill2', 3)

In [ ]:
# 밀린 뒤 비게 된 마지막 열(WorkMode) 채우기

# 학습 자료
train_df['WorkMode Collect Result_Fill1'] = 7
train_df['WorkMode Collect Result_Dam'] = 7
train_df['WorkMode Collect Result_Fill2'] = 0

# 시험 자료
test_df['WorkMode Collect Result_Fill1'] = 7
test_df['WorkMode Collect Result_Dam'] = 7
test_df['WorkMode Collect Result_Fill2'] = 0

### 2.2. 판정 변수 인코딩

- 판정 결과 변수: OK → 1, NG와 결측 → 0

In [ ]:
def process_ok_columns(df, col):
    df[col].replace('OK', 1, inplace=True)

    df[col].fillna(0, inplace=True)

    df[col].replace('NG', 0, inplace=True)

cols = ['Chamber Temp. Judge Value_AutoClave', 'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave']
process_ok_columns(train_df, cols[0])
process_ok_columns(test_df,  cols[0])
process_ok_columns(train_df, cols[1])
process_ok_columns(test_df,  cols[1])

### 2.3. 장비 인코딩과 공정 간 불일치 확인

- 장비 이름: #1 → 0, #2 → 1
- 여러 공정에 반복 기록된 식별 변수(Model.Suffix, Workorder, Receip No, PalletID, Production Qty, Equipment)의 공정 간 불일치 행 확인

In [ ]:
# 1) 장비 인코딩: #1 → 0, #2 → 1
def convert_equipment_values(row):
    if '#1' in str(row):
        return 0
    elif '#2' in str(row):
        return 1
    return row  # 둘 다 아니면 원래 값

train_df['Equipment_Dam'] = train_df['Equipment_Dam'].apply(convert_equipment_values)
train_df['Equipment_Fill1'] = train_df['Equipment_Fill1'].apply(convert_equipment_values)
train_df['Equipment_Fill2'] = train_df['Equipment_Fill2'].apply(convert_equipment_values)

test_df['Equipment_Dam'] = test_df['Equipment_Dam'].apply(convert_equipment_values)
test_df['Equipment_Fill1'] = test_df['Equipment_Fill1'].apply(convert_equipment_values)
test_df['Equipment_Fill2'] = test_df['Equipment_Fill2'].apply(convert_equipment_values)

# 공정마다 반복 기록된 변수 묶음
equipment_cols = ['Equipment_Dam', 'Equipment_Fill1', 'Equipment_Fill2']
model_suffix_cols = [col for col in train_df.columns if "Model.Suffix" in col]
workorder_cols = [col for col in train_df.columns if "Workorder" in col]
receip_no_cols = [col for col in train_df.columns if "Receip No Collect Result" in col]
pallet_id_cols = [col for col in train_df.columns if "PalletID" in col]
production_qty_cols = [col for col in train_df.columns if "Production" in col]

columns_groups = {
    'Model Suffix': model_suffix_cols,
    'Workorder': workorder_cols,
    'Receip No': receip_no_cols,
    'Pallet ID': pallet_id_cols,
    'Production Qty': production_qty_cols,
    'Equipment': equipment_cols
}


# 2) 공정 간 값이 다른 행 찾기
def find_differences(data, columns):
    if not columns:
        return None
    comparison_df = data[columns]
    diff_rows = comparison_df[comparison_df.nunique(axis=1) > 1]  # 행 안에서 값이 2개 이상이면 불일치
    print(f'✅ <불일치한 경우> 총 {len(diff_rows)}개 rows가 불일치함. 비교한 {len(diff_rows.columns)}개 columns 정보: {diff_rows.columns.to_list()}')
    return

def remove_inconsistent_rows(data, columns_groups):
    for key, cols in columns_groups.items():
        diff_rows = find_differences(data, cols)
        if diff_rows is not None:
            data.drop(diff_rows.index, inplace=True)
    return data

train_df = remove_inconsistent_rows(train_df, columns_groups)


# 3) 중복 열 제거: 이후 변수 선택 단계에서 필요한 열만 남기므로 실행하지 않음
'''
remove_cols = ['Model.Suffix_AutoClave', 'Model.Suffix_Fill1', 'Model.Suffix_Fill2',
               'Equipment_Fill1', 'Equipment_Fill2',
               'Workorder_AutoClave', 'Workorder_Fill1', 'Workorder_Fill2',
               'Receip No Collect Result_Fill1', 'Receip No Collect Result_Fill2',
               'Production Qty Collect Result_Fill1', 'Production Qty Collect Result_Fill2',
               'PalletID Collect Result_Fill1','PalletID Collect Result_Fill2']

print("삭제 전:")
print(train_df.shape)
print(test_df.shape)

train_df = train_df.drop(columns=remove_cols)
test_df  = test_df.drop(columns=remove_cols)

print("삭제 후:")
print(train_df.shape)
print(test_df.shape)
'''

## 3. 탐색

- 모든 값이 결측인 변수, 값이 하나뿐인 변수 제외 후 변수별 빈도 확인

In [ ]:
train_data = train_df.copy()  # 탐색용 사본

In [ ]:
# 모든 값이 결측인 변수

nan_columns = train_data.columns[train_data.isna().all()]
nan_columns

In [ ]:
train_data = train_data.drop(nan_columns, axis=1)
train_data

In [ ]:
# 값이 하나뿐인 변수

single_value_columns = train_data.columns[train_data.nunique() == 1]
len(single_value_columns)

In [ ]:
train_data = train_data.drop(single_value_columns, axis=1)
train_data

In [ ]:
for z, i in enumerate(train_data.columns):
    print(train_data[i].value_counts())
    print(len(train_data[i].value_counts()))
    print('++++'*5, f'num : {z+1}', '\n')

### 3.1. 남은 변수 정리


### Dam
1. Equipment_Dam : 공정 장비 [Dam dispenser #1, Dam dispenser #2]

2. Model.Suffix_Dam : 장비 종류 [7 범주]

3. Workorder_Dam : 식별 코드 [len : 663] (파생변수 후보)

4. CURE END POSITION X Collect Result_Dam : 경화 종료 X 위치 [len : 2, (240, 1000)]

5. CURE END POSITION Z Collect Result_Dam : 경화 종료 Z 위치 [len : 2, (2.5, 12.5)]

6. CURE END POSITION Θ Collect Result_Dam :  // [len :2, (-90, 90)]

7. CURE SPEED Collect Result_Dam : 경화 속도 [len : 5]

8. CURE START POSITION X Collect Result_Dam : 경화 시작 X 위치 [len : 2. (280, 1030)]

9. CURE START POSITION Θ Collect Result_Dam : // [len : 2, (-90, 90)]

10. DISCHARGED SPEED OF RESIN Collect Result_Dam : 수지 방출 속도 [len : 3]

11. DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam : 수지 방출 시간 s1 [연속]
12. //(s2)
13. //(s3)

14. Dispense Volume(Stage1) Collect Result_Dam : 수지 방출량 s1 [연속]
15. //(s2)
16. //(s3)

17. HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam : X 헤드 위치  s1 [연속 ! OK라는 오류항 존재]
18. //(s2)
19. //(s3)

20. HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Dam : Y 헤드 위치 s1 [연속]
21. //(s2)
22. //(s3)

23. HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Dam : Z 헤드 위치 s1 [연속]
24. //(s2)
25. //(s3)

26. HEAD Standby Position X Collect Result_Damv : X 헤드 대기 위치 [연속]
27. // : Y 헤드 대기 위치 [len : 2 (257, 66)]
28. // : Z 헤드 대기 위치 [len : 2 (66, 0)]

29. Head Clean Position X Collect Result_Dam : X 헤드 세척 위치 [len :2 (66, 0)]
30. Head Clean Position Y Collect Result_Dam : Y 헤드 세척 위치 [len :2 (127, 66)]
31. Head Clean Position Z Collect Result_Dam : Z 헤드 세척 위치 [len : 4]

32. Head Purge Position X Collect Result_Dam : X 헤드 찌꺼지 배출 위치 [len ; 4]
33. Head Purge Position Y Collect Result_Dam : Y 헤드 찌꺼지 배출 위치 [len ; 2 (257, 66)]
34. Head Purge Position Z Collect Result_Dam : Z 헤드 찌꺼지 배출 위치 [len ; 4]

35. Head Zero Position X Collect Result_Dam : X 헤드 기본 위치 [len : 2 (130.85, 505)]
36. Head Zero Position Y Collect Result_Dam : X 헤드 기본 위치 [len : 3]
37. Head Zero Position Z Collect Result_Dam : X 헤드 기본 위치 [len : 3]

38. Machine Tact time Collect Result_Dam : 사이클 완료 시간 [연속]

39. PalletID Collect Result_Dam : 팔레트 식별 번호 [len : 374]

40. Production Qty Collect Result_Da : 생산 수량 [연속]

41. Receip No Collect Result_Dam : 영수증 번호 [len : 607]

42. Stage1 Circle1 Distance Speed Collect Result_Dam : CID #1 R 수치 [len : 8 (1, 3, 6, 7, 17, 4000, 5000, 9000]
43. Stage1 Circle2 Distance Speed Collect Result_Dam : CID #2 R 수치 [len : 7 (4000~9000)]
44. Stage1 Circle3 Distance Speed Collect Result_Dam : CID #3 R 수치 [len : 7 (4000~9000)]
45. Stage1 Circle4 Distance Speed Collect Result_Dam : CID #4 R 수치 [len : 7 (4000~9000)]
> 43~45 값이 비슷해 상관 확인 필요

46. Stage1 Line1 Distance Speed Collect Result_Dam : CID #1 토출 스피드 [len : 8 (4000~9000)]
47. Stage1 Line2 Distance Speed Collect Result_Dam : CID #2 토출 스피드 [len : 8 (4000~9000)]
48. Stage1 Line3 Distance Speed Collect Result_Dam : CID #3 토출 스피드 [len : 8 (4000~9000)]
49. Stage1 Line4 Distance Speed Collect Result_Dam : CID #4 토출 스피드 [len : 8 (4000~9000)]
> 46~49도 값이 비슷함

50~65. Stage2, Stage3 반복

66. THICKNESS 1 Collect Result_Dam : 두께 1 [len : 6 (0값도 존재)]
67. THICKNESS 2 Collect Result_Dam : 두께 2 [len : 6 (0값도 존재)]
68. THICKNESS 3 Collect Result_Dam : 두께 3 [len : 7 (0값도 존재)]

69. WorkMode Collect Result_Dam : [len : 8]

### AutoClave

70. Model.Suffix_AutoClave
71. Workorder_AutoClave

72. 1st Pressure Collect Reulst_AutoClave : 1st 탈포 압력 [연속]
73. 1st Pressure 1st Pressure Unit Time_AutoClave : 1st 탈포 시간 [연속] - feature name 오류 -

74. 2st Pressure Collect Reulst_AutoClave : 2st 탈포 압력 [연속]
75. 2st Pressure Unit Time_AutoClave : 2st 탈포 시간 [연속]

76. 3st Pressure Collect Reulst_AutoClave : 3st 탈포 압력 [연속]
77. 3st Pressure Unit Time_AutoClave : 3st 탈포 시간 [연속]

78. Chamber Temp. Collect Result_AutoClave :챔버 온도 [연속]
79. Chamber Temp. Unit Time_AutoClave : 공정 소요 시간 [연속]
80. Chamber Temp. Judge Value_AutoClave : 판정값 [OK or NG]

### Fill1
81, 82, 83. Equipment, Model.Suffix, Workorder

84. DISCHARGED SPEED OF RESIN Collect Result_Fill : 토출 속도 [len : 4 , 0값 한개]
85. DISCHARGED TIME OF RESIN(Stage1) Collect Result_Fill1 : 토출 시간 s1 [연속, 0값 한개]
86. DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1 : 토출 시간 s2 [연속, 0값 한개]
87. DISCHARGED TIME OF RESIN(Stage3) Collect Result_Fill1 : 토출 시간 s3 [연속, 0값 한개]

88. Dispense Volume(Stage1) Collect Result_Fill1 : 토출량 s1 [연속, 0값 한개]
89. Dispense Volume(Stage2) Collect Result_Fill1 : 토출량 s2 [연속, 0값 한개]
90. Dispense Volume(Stage3) Collect Result_Fill1 : 토출량 s3 [연속, 0값 한개]

91. HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1 : OK항 존재
92. HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Fill1 : [연속]
93. HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill1 : [연속]

94. HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill1 : [연속]
95. HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Fill1 : [연속]
96. HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill1 : [연속]

97. HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1 ; [연속]
98. HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1 ; [연속]
99. HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1 ; [연속]

100. HEAD Standby Position X Collect Result_Fill1 : [연속]
101. HEAD Standby Position Y Collect Result_Fill1 : [len 2: (289, 50)]
102. HEAD Standby Position Z Collect Result_Fill1 : [len 2 : (50, 0)]
> Y, Z 비율 같음: 변수 통합 후보

103. Head Clean Position X Collect Result_Fill1 : [len 2 : (0.0, 123.4)]
104. Head Clean Position Y Collect Result_Fill1 : [len 2 : (123.4, 50.0)]
105. Head Clean Position Z Collect Result_Fill1 : [len 2 : (0.0, 123.4)]
> X, Y, Z 비율 같음

106. Head Purge Position X Collect Result_Fill1 : [len 2 : (92.2, 289,0)]
107. Head Purge Position Y Collect Result_Fill1 : [len 2 : (289, 50)]
108. Head Purge Position Z Collect Result_Fill1 : [len 3 : (50, 85, 145)]

109, 110, 111, 112, 113. Machine Tact Time, PalletID, Production Qty, Recip No, WorkMode 

### Fill2

114, 115, 116. Equipment, Model.Suffix, Workorder

117. CURE END POSITION X Collect Result_Fill2 : [len 2 :(240, 1020)]
118. CURE END POSITION Z Collect Result_Fill2 : [len 3]

119. CURE SPEED Collect Result_Fill2 : [연속]

120. CURE STANDBY POSITION Z Collect Result_Fill2 : [len 4]

121. CURE START POSITION X Collect Result_Fill2 : [len 2 : (1020, 240)]
> 107과 비율 같음

122. CURE START POSITION Z Collect Result_Fill2 : [len 4]

123. HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2 : [len 3] ! OK 항 있음
124. HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Fill2 : [len 5]
> 113의 OK를 제외하면 비율이 같음: OK 대체 후보

125. HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill2 : [len 4]

126. HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill2 : [len : 5]
127. HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Fill2 : [len : 3]
128. HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill2 : [len : 3]
> 117, 118 비율 같음

129. HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill2 : [len : 4]
130. HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill2 : [len : 2]
131. HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Fill2 : [len : 2]
> 120, 121 비율, 값 같음

132. HEAD Standby Position X Collect Result_Fill2 : [len : 3]
133. HEAD Standby Position Y Collect Result_Fill2 : [len : 2]
134. HEAD Standby Position Z Collect Result_Fill2 : [len : 2]
> 123, 124 비율 같음

135. Head Clean Position X Collect Result_Fill2 : [len : 2]
136. Head Clean Position Y Collect Result_Fill2 : [len : 2]
137. Head Clean Position Z Collect Result_Fill2 : [len : 2]
> 125, 126, 127 비율 같음

138. Head Purge Position X Collect Result_Fill2 : [len : 2]
139. Head Purge Position Y Collect Result_Fill2 : [len : 2]
140. Head Purge Position Z Collect Result_Fill2 : [len : 2]
> 128, 129, 130 비율 같음

141, 142, 143, 144, 145. Machine Tact Time, PalletID, Production Qty, Recip No, WorkMode 

146. target : [len 2 : (38156, 2350)]

### 3.2. Dam

In [ ]:
Dam_data = train_data.filter(like = 'Dam')
print(f'Dam shape : {Dam_data.shape}')
Dam_data.describe()

In [ ]:
def plot_dist(X, Y, data):
    i = 0
    plt.figure(figsize=(25, 20))
    plt.subplots(Y, X, figsize=(25, 20))
    
    for c in data.columns:
        i += 1
        plt.subplot(Y, X, i)
        
        if pd.api.types.is_numeric_dtype(data[c]):
            sns.histplot(data[c], kde=True)
        else:
            sns.countplot(y=data[c])
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_dist(7, 10, Dam_data)

In [ ]:
sns.set(font_scale=0.8)
plt.figure(figsize=(30, 20))

numeric_cols = Dam_data.select_dtypes(include='number')

sns.heatmap(numeric_cols.corr(method='pearson'), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
def find_pair(data):
    num_data = data[data.describe().columns]
    cor_mat = num_data.corr()
    perfect_cor = []

    for i in range(len(cor_mat)):
        for j in range(i + 1, len(cor_mat)):
            if np.isclose(cor_mat.iloc[i, j], 1.0):
                perfect_cor.append((cor_mat.index[i], cor_mat.columns[j]))

    for k in perfect_cor:
        print(k)
    return perfect_cor

In [ ]:
perfect_cor = find_pair(Dam_data)

In [ ]:
def check_pair(list, data):
    per_cor_variables = []
    i = 0
    print('===' * 5, 'check pair')
    for j in list:
        sum_variable = data[j[0]] + data[j[1]]
        if len(np.unique(sum_variable)) == 2:
            print('OK', end = ' ')
            per_cor_variables.append(j)
        else:
            i += 1
            print('NG', end = ' ')
    
    print('\n','===' * 5, f'Whole : {len(list)}',  f'/ num of NG : {i}')
    return per_cor_variables

per_cor_variables = check_pair(perfect_cor, Dam_data)

In [ ]:
# 완전 상관(r = 1) 변수 쌍
unique_variables = set()

for pair in per_cor_variables:
    unique_variables.update(pair)
    unique_variables_list = list(unique_variables)
unique_variables_list

In [ ]:
plot_dist(4, 3, Dam_data[unique_variables_list])

In [ ]:
per_cor_variables

In [ ]:
# 완전 상관 쌍 중 한쪽 제거
del_col = ['CURE END POSITION Z Collect Result_Dam',
          'CURE END POSITION Θ Collect Result_Dam',
          'CURE START POSITION Θ Collect Result_Dam',
          'HEAD Standby Position Z Collect Result_Dam',
          'Head Clean Position Y Collect Result_Dam',
          'Head Purge Position Y Collect Result_Dam',
          'Head Zero Position X Collect Result_Dam']

Dam_data = Dam_data.drop(del_col, axis=1)
Dam_data

결과변수(정상, 불량)와의 관계

In [ ]:
Dam_data['target'] = train_data['target']

def target_plot(data):
    for feature in data.columns:
        if feature == 'target':
            continue

        plt.figure(figsize=(8, 6))
        
        if pd.api.types.is_numeric_dtype(data[feature]):
            # 연속형: 정상과 불량 집단의 분포
            sns.histplot(data[data['target'] == 'Normal'][feature], color='blue', label='Normal', kde=True, alpha=0.5)
            sns.histplot(data[data['target'] == 'AbNormal'][feature], color='red', label='AbNormal', kde=True, alpha=0.5)
            plt.title(f'Distribution of {feature} by Target')
            plt.xlabel(feature)
            plt.ylabel('Frequency')
        
        else:
            # 범주형: 범주별 정상, 불량 빈도
            category_counts = data.groupby([feature, 'target']).size().reset_index(name='count')
            sns.barplot(x=feature, y='count', hue='target', data=category_counts, alpha=0.7)
            plt.title(f'Bar Plot of {feature} by Target')
            plt.ylabel('Count')

        plt.legend()
        plt.show()

In [ ]:
target_plot(Dam_data)

In [ ]:
def calculate_proportion(data):
    proportion_dict = {}

    # 결과변수 집단 안에서 범주별 비율
    for feature in data.columns:
        if feature == 'target':
            continue

        grouped = data.groupby(['target', feature]).size()

        proportion = grouped / grouped.groupby(level=0).sum()

        proportion_df = proportion.reset_index(name='Proportion')
        pivot_df = proportion_df.pivot(index=feature, columns='target', values='Proportion').fillna(0)

        proportion_dict[feature] = pivot_df

    return proportion_dict

proportion_results = calculate_proportion(Dam_data)

for feature, df in proportion_results.items():
    print(f"Feature: {feature}")
    print(df)
    print("\n")

- Dam의 HEAD NORMAL COORDINATE X AXIS(Stage1): OK(결측) 외 548.5 값의 비율이 정상과 불량 집단에서 약 0.13 차이 나므로 변수 유지 검토

### 3.3. AutoClave

In [ ]:
AC_data = train_data.filter(like = 'AutoClave')
print(f'AC shape : {AC_data.shape}')
AC_data.describe()

In [ ]:
plot_dist(4, 3, AC_data)

In [ ]:
sns.set(font_scale=2.5)
plt.figure(figsize=(30, 20))

numeric_cols = AC_data.select_dtypes(include='number')

sns.heatmap(numeric_cols.corr(method='pearson'), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')

결과변수(정상, 불량)와의 관계

In [ ]:
AC_data['target'] = train_data['target']

sns.set(font_scale=1)
target_plot(AC_data)

In [ ]:
proportion_results = calculate_proportion(AC_data)

for feature, df in proportion_results.items():
    print(f"Feature: {feature}")
    print(df)
    print("\n")

### 3.4. Fill1

In [ ]:
F1_data = train_data.filter(like = 'Fill1')
print(f'F1 shape : {F1_data.shape}')
F1_data.describe()

In [ ]:
plot_dist(6, 6, F1_data)

In [ ]:
sns.set(font_scale=1)
plt.figure(figsize=(30, 20))

numeric_cols = F1_data.select_dtypes(include='number')

sns.heatmap(numeric_cols.corr(method='pearson'), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')

In [ ]:
perfect_cor = find_pair(F1_data)

In [ ]:
per_cor_variables = check_pair(perfect_cor, F1_data)

In [ ]:
# 완전 상관(r = 1) 변수 쌍
unique_variables = set()

for pair in per_cor_variables:
    unique_variables.update(pair)
    unique_variables_list = list(unique_variables)
unique_variables_list

In [ ]:
plot_dist(4, 2, F1_data[unique_variables_list])

In [ ]:
# 완전 상관 쌍 중 한쪽 제거
del_col = ['HEAD Standby Position Z Collect Result_Fill1',
          'Head Clean Position Y Collect Result_Fill1',
          'Head Purge Position Y Collect Result_Fill1',
          'Head Clean Position Z Collect Result_Fill1',
          'Head Purge Position X Collect Result_Fill1']

F1_data = F1_data.drop(del_col, axis=1)

결과변수(정상, 불량)와의 관계

In [ ]:
F1_data['target'] = train_data['target']

target_plot(F1_data)

In [ ]:
proportion_results = calculate_proportion(F1_data)

for feature, df in proportion_results.items():
    print(f"Feature: {feature}")
    print(df)
    print("\n")

### 3.5. Fill2

In [ ]:
F2_data = train_data.filter(like = 'Fill2')
print(f'F2 shape : {F2_data.shape}')
F2_data.describe()

In [ ]:
plot_dist(6, 6, F2_data)

In [ ]:
sns.set(font_scale=1)
plt.figure(figsize=(30, 20))

numeric_cols = F2_data.select_dtypes(include='number')

sns.heatmap(numeric_cols.corr(method='pearson'), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')

In [ ]:
perfect_cor = find_pair(F2_data)

In [ ]:
per_cor_variables = check_pair(perfect_cor, F2_data)

In [ ]:
# 완전 상관(r = 1) 변수 쌍
unique_variables = set()

for pair in per_cor_variables:
    unique_variables.update(pair)
    unique_variables_list = list(unique_variables)
unique_variables_list

In [ ]:
plot_dist(4, 3, F2_data[unique_variables_list])

In [ ]:
per_cor_variables

In [ ]:
# 완전 상관 쌍 중 한쪽 제거
del_col = ['HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill2',
          'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Fill2',
          'HEAD Standby Position Z Collect Result_Fill2',
          'Head Clean Position Y Collect Result_Fill2',
          'Head Purge Position Y Collect Result_Fill2',
          'Head Clean Position Z Collect Result_Fill2',
          'Head Purge Position X Collect Result_Fill2',
          'Head Purge Position Z Collect Result_Fill2']

F2_data = F2_data.drop(del_col, axis=1)

결과변수(정상, 불량)와의 관계

In [ ]:
F2_data['target'] = train_data['target']

target_plot(F2_data)

In [ ]:
proportion_results = calculate_proportion(F2_data)

for feature, df in proportion_results.items():
    print(f"Feature: {feature}")
    print(df)
    print("\n")